# Phase 0 — Smoke Gate

**EMNLP topic-domination analysis** · Primary run: `data/outputs/20260211_2122`

This notebook validates that the primary evolution run is ready for Phases 1–10:

| Check | Threshold |
|-------|-----------|
| Unified genome count \|G\| | ≥ 5,000 |
| Topics with ≥5 genomes | ≥ 5 |
| Complete 8-D moderation scores | ≥ 95% of genomes |
| Required population files | `elites.json`, `reserves.json`, `archive.json` |

**Outputs:** `experiments/emnlp_topic_domination/outputs/gate0_smoke.json`

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd


def find_analysis_root() -> Path:
    here = Path.cwd().resolve()
    candidates = [here]
    if here.name == "notebooks":
        candidates.append(here.parent)
    candidates.extend(here.parents)
    for base in candidates:
        root = base if (base / "analysis_utils.py").is_file() else base / "experiments" / "emnlp_topic_domination"
        if (root / "analysis_utils.py").is_file():
            return root
    raise RuntimeError("Could not locate experiments/emnlp_topic_domination/analysis_utils.py")


ANALYSIS_ROOT = find_analysis_root()
REPO_ROOT = ANALYSIS_ROOT.parents[1]
sys.path.insert(0, str(ANALYSIS_ROOT))

from analysis_utils import (
    ANALYSIS_OUTPUT_ROOT,
    DEFAULT_PRIMARY_RUN,
    PERSPECTIVE_AXIS_ORDER,
    smoke_validate_run,
)

# --- configurable ---
RUN_PATH = DEFAULT_PRIMARY_RUN
MIN_GENOMES = 5_000
MIN_TOPICS = 5
MIN_TOPIC_SIZE = 5
MIN_OBJECTIVE_FRAC = 0.95

OUTPUT_JSON = ANALYSIS_OUTPUT_ROOT / "gate0_smoke.json"

print(f"Analysis root: {ANALYSIS_ROOT}")
print(f"Repo root:     {REPO_ROOT}")
print(f"Primary run:   {RUN_PATH}")
print(f"Output:        {OUTPUT_JSON}")
print(f"Axis order:    {PERSPECTIVE_AXIS_ORDER}")

In [ ]:
gate0 = smoke_validate_run(
    RUN_PATH,
    min_genomes=MIN_GENOMES,
    min_topics=MIN_TOPICS,
    min_topic_size=MIN_TOPIC_SIZE,
    min_objective_frac=MIN_OBJECTIVE_FRAC,
)

status = "PASS" if gate0["pass"] else "FAIL"
print(f"\nGate 0: {status}\n")

checks_df = pd.DataFrame(
    [
        {"check": name, "passed": passed}
        for name, passed in gate0["checks"].items()
    ]
)
display(checks_df)

summary = {
    "n_genomes": gate0["n_genomes"],
    "n_species": gate0["n_species"],
    "n_large_topics": gate0["n_large_topics"],
    "frac_with_objectives": round(gate0["frac_with_objectives"], 4),
    "n_with_embedding": gate0["n_with_embedding"],
    "n_duplicates_dropped": gate0["n_duplicates_dropped"],
}
display(pd.DataFrame([summary]))

In [ ]:
files_df = pd.DataFrame(
    [
        {
            "file": fname,
            "present": gate0["files_present"].get(fname, False),
            "n_records": gate0["files"].get(fname, 0),
        }
        for fname in ("elites.json", "reserves.json", "archive.json", "temp.json")
    ]
)
display(files_df)

if gate0["missing_files"]:
    print("Missing required files:", gate0["missing_files"])

topic_rows = [
    {"species_id": sid, "n_genomes": n, "large_enough": n >= MIN_TOPIC_SIZE}
    for sid, n in gate0["topic_sizes"].items()
]
topics_df = pd.DataFrame(topic_rows).sort_values("n_genomes", ascending=False)
display(topics_df)

In [ ]:
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON.write_text(json.dumps(gate0, indent=2), encoding="utf-8")
print(f"Wrote {OUTPUT_JSON}")

if not gate0["pass"]:
    failed = [k for k, v in gate0["checks"].items() if not v]
    raise SystemExit(f"Gate 0 FAILED — failed checks: {failed}")